# 11 Regression V2 — 적정가 예측 · 오차 범위 · 적정가 대비 등급 · OOF Gap

## 목적

| 결과 | 쓰임 |
|---|---|
| **적정가** (거래 단위 예측가) | 웹 단지 화면의 "모델 적정가" |
| **80% 오차 범위** | "적정가 범위" 표시, 등급 판정 기준 |
| **적정가 대비 등급** (규칙) | 저평가 ~ 고평가 5단계 (모델 기준 표현, 매수 · 매도 권유 없음) |
| **OOF Gap 구 × 월** | 31 분류 Feature 후보 (회귀 → 분류 연결) |

## 결론 요약

- **모델:** LightGBM 단일 모델 (Optuna 튜닝), 정답 `log(거래금액)`, 기본 Feature 201개 (`v2_feature_registry.csv` role=model).
- **평가는 서비스 방식(매달 재학습)을 기준으로 한다.** 매달 m월 시작 시점에 확정된 거래(m-2월까지)로 다시 학습해 m월을 예측한다.
- **기준선은 사용자가 이미 보는 값 — 같은 단지 · 같은 면적 직전 실거래가**다. 첫 버전은 이 기준선과 거의 같아서(중앙 오차 5.23 vs 5.26%),
  직전 거래 Feature 5개를 06에 추가하고 다시 튜닝했다 (5-2장).
  이어서 직전 거래 뒤 시세 변화를 구 주간 지수로 보정한 2개를 더했다 (5-3장, 보완 2).

| 지표 (매달 재학습, 2026-01~06 Test, 중개거래) | 값 |
|---|---:|
| MAPE / 중앙 오차 (전체 중개거래) | **5.35% / 3.95%** (보완 2 전 5.53 / 4.22%) |
| 직전 거래가 있는 거래(97.7%): 모델 vs 같은 단지 · 면적 직전 거래가 | **5.19% vs 6.67%** (중앙 3.89 vs 4.84%) |
| 직전 거래가 × 시세 보정 (모델 없이) | 6.23% |
| 같은 기간 단순 규칙 (단지 최근 12개월 평당가 × 평) | 14.94% |
| 80% 구간 포함률 / 평균 폭 (기존 단지) | 81.8% / 16.4% (7장) |

- 고정 분할(2025-06까지 학습 → 2026 상반기)로는 보완 전 모델 Test MAPE 9.64%다. 6~12개월 재학습 없이 오른 시장을 따라가지 못한 값이다.
  같은 조건에서 V1 Feature는 13.26%, V2 Feature는 9.92%다.
- 앙상블 · 정답 변환 · 최근 오차 보정은 **채택 기준(두 fold 모두 개선, 홀드아웃 악화 없음)**을 넘지 못해 쓰지 않는다.
- 신규 단지(거래 이력 없음)는 구간을 넓혀도 80%를 맞추지 못해 **등급을 매기지 않고 범위만 참고용으로** 보여준다.

## 원칙

- 분할: Train 2021-07~2025-06, Valid 2025-07~12, Test 2026-01~06 (V1과 동일). 선택은 Train/Valid로만 하고 Test는 확인용이다.
- 실험(탐색 · 재학습 수 시간)은 `v2_experiments/` 스크립트로 돌렸고, 이 노트북은 그 결과를 불러와 해석한다.
  오차 범위 · 등급 · OOF Gap · 최종 모델은 여기서 직접 계산한다.
- Seed만 바꿔도 Valid MAPE가 6.34~6.78%로 흔들렸다 (학습률 0.1). **0.4%p보다 작은 차이는 우연일 수 있다**고 보고 판단한다.
- 서비스 기준 모집단은 **중개거래**다. 직거래는 가족 간 거래 등 시세와 다른 가격이 섞여 오차가 2배다 (따로 보고).

In [1]:
from pathlib import Path
import gzip
import json
import sqlite3

import lightgbm as lgb
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)


def find_project_root(start=Path.cwd()):
    for path in [start, *start.parents]:
        if (path / "data").exists() and (path / ".env").exists():
            return path
    raise FileNotFoundError("프로젝트 루트를 찾을 수 없습니다.")


PROJECT_DIR = find_project_root()
PROCESSED_DIR = PROJECT_DIR / "data" / "v2_5year_seoul" / "processed"
EXP_DIR = PROJECT_DIR / "notebook" / "v2_5year_notebook" / "model" / "regression" / "v2_experiments"
MODEL_DIR = PROJECT_DIR / "models" / "regression" / "v2"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

registry = pd.read_csv(PROCESSED_DIR / "v2_feature_registry.csv")
registry = registry[registry["dataset"].eq("v2_trade")]
FEATURES = registry.loc[registry["role"].eq("model"), "column"].tolist()
trade = pd.read_parquet(PROCESSED_DIR / "v2_trade.parquet")
trade["단지일련번호"] = trade["단지일련번호"].astype(str)
CATEGORICAL = [c for c in FEATURES if trade[c].dtype == object]

print("기본 Feature", len(FEATURES), "| 범주형", CATEGORICAL)
print(trade.groupby("데이터구분")["거래월일자"].agg(["min", "max", "count"]))

기본 Feature 201 | 범주형 ['거래유형', '토지임대부아파트여부']
             min        max   count
데이터구분                              
test  2026-01-01 2026-06-01   39569
train 2021-07-01 2025-06-01  160945
valid 2025-07-01 2025-12-01   33788


## 1. 기준선과 Feature 묶음 비교

같은 빠른 설정(XGBoost, 학습률 0.1, 깊이 6)으로 Feature 묶음을 하나씩 더했다 (`r11_stage1.py`).

- **Seen/Unseen:** Valid 거래 중 Train에 있던 단지 / Valid에서 처음 거래된 단지(950건).
- **단지 홀드아웃:** Train 단지 20%를 학습에서 빼고, 그 단지들의 Valid 거래로 평가 (3회 평균). 거래 이력은 있지만 모델이 처음 보는 단지다.
- **단순 규칙:** 같은 단지 확정 12개월(m-13~m-2) 중위 평당가 × 평, 없으면 동 → 구.

In [2]:
stage1 = pd.read_csv(EXP_DIR / "r11_stage1.csv")
naive = json.load(open(EXP_DIR / "r11_naive.json", encoding="utf-8"))
print("단순 규칙:", {k: round(v, 2) if isinstance(v, float) else v for k, v in naive.items()})
stage1.drop(columns=["초"]).round(2)

단순 규칙: {'구성': '단순 규칙 (단지 확정 12개월 중위 평당가 × 평)', 'Valid_MAPE': 13.45, 'Test_MAPE': 14.94, 'Seen_MAPE': 12.72, 'Unseen(Valid 신규)_MAPE': 38.73, '단지값 사용 비율(Valid)': 0.96}


,구성,Feature수,best_iter,Valid_MAPE,Valid_중앙APE,Valid_10%이내,Seen_MAPE,Unseen_MAPE(Valid 신규),홀드아웃_MAPE,홀드아웃_MAPE_범위
0,V1raw (V1 시점 그대로),56,1612,9.25,7.06,64.19,9.11,13.97,12.42,11.92~13.02
1,F0 V1정의+V2시점 (출처·응급·유치원·학교거리 포함),60,2338,8.64,6.40,67.95,8.43,15.59,12.22,11.79~12.59
2,F1 핵심 (레지스트리 기본),52,1778,9.14,6.80,65.79,8.89,17.58,12.59,11.84~13.36
3,F1+공시,57,1437,6.71,5.11,79.52,6.63,9.47,7.27,7.21~7.34
4,F2 +단지속성,68,1671,8.05,6.07,70.54,7.89,13.61,10.71,10.41~11.16
5,F3 +단지속성+공시,73,1515,6.57,5.05,80.48,6.50,8.97,7.12,6.98~7.24
6,F4 +학교·학원,102,1952,6.51,4.91,80.58,6.39,10.60,7.20,7.16~7.22
7,F5 +구×월 (전체 196),196,1155,6.48,4.84,80.85,6.25,14.31,6.91,6.79~7.08
8,F6 +군집 라벨,198,883,6.50,4.90,80.52,6.32,12.76,6.97,6.94~6.99


- **시점을 바로잡아도 성능이 떨어지지 않는다.** 같은 V1 Feature에서 V1 시점 9.25% → V2 시점(확정월) 8.64%.
- **개선의 대부분은 공시가격이다.** 핵심 9.14% → +공시 6.71%, 단지 홀드아웃 12.59% → 7.27%.
  `공시기반_추정가`는 공시 ㎡당 가격 × 전용면적이라 실거래가가 섞이지 않고, 계약일 기준 이미 공시된 연도만 쓴다
  (2025-06 거래 → 2024년 공시, 2025-07 거래 → 2025년 공시, 05b · 06에서 검증).
- **군집 라벨은 효과가 없다** (6.48 → 6.50). V1과 같은 결론이라 쓰지 않는다.
- **새 문제:** 묶음을 더할수록 Valid 신규 단지 오차가 커진다 (단지속성+공시 8.97% → +학교 10.60% → +구×월 14.31%) → 2장.

## 2. Ablation — 예상가격 대체값 문제

`r11_check.py`로 Valid 신규 단지를 보면 66%가 준공 1년 이내, 64%가 공시가격이 아직 없다.
이 단지들의 `예상가격_만원`은 단지 이력이 없어 **동 · 면적대 → 구 중위값으로 대체**된다. 신축은 주변 구축보다 비싸서 대체값이 기준가를 틀리게 준다.

In [3]:
print(open(EXP_DIR / "r11_check_output.txt", encoding="utf-8").read().split("묶음 permutation")[0])
ablation = pd.read_csv(EXP_DIR / "r11_ablation.csv").drop(columns=["초"])
abl2 = pd.read_csv(EXP_DIR / "r11_abl2.csv")
display(ablation.round(2))
abl2.round(2)

Valid 신규 단지 거래 950 | 단지 293 | 연식 중앙 0.0 | 신축(연식<=1) 비율 0.661 | 거래유형 {'직거래': 0.65, '중개거래': 0.35}
공시 결측 비율(신규) 0.642 | 전체 Valid 0.027
F3 +단지속성+공시 신규 단지 APE 중앙 6.61 평균 8.97 | 상위 5% 제외 평균 7.48
F5 +구×월 (전체 196) 신규 단지 APE 중앙 11.68 평균 14.31 | 상위 5% 제외 평균 12.87
         F3           F5       
      count   mean count   mean
공시없음                           
False   340  11.72   340  11.82
True    610   7.44   610  15.70
                       F3     F5  diff
단지            구                       
용산남영역롯데캐슬헤리티지 용산구     3.4   16.5  13.1
계명(350-34)    강서구    12.8   26.3  13.5
라봄성동          성동구     3.9   18.2  14.3
(1-10)        성동구   116.9  134.9  18.0
고명            성북구    18.1   36.3  18.3
LOFT129       동대문구    6.4   30.3  24.0
이수스위첸포레힐즈     동작구     3.7   28.0  24.3
글로리안(A)       중구     52.4   90.1  37.7



,구성,Feature수,Valid_MAPE,Valid_중앙APE,Seen_MAPE,Valid신규_MAPE,홀드아웃_MAPE
0,A0 전체(196),196,6.48,4.84,6.25,14.31,6.91
1,A1 예상가격 계열 제외,193,6.77,5.12,6.66,10.42,7.31
2,A2 + 예상가격_출처,197,6.87,5.25,6.71,12.16,6.95
3,A3 + 전세피처출처,199,6.50,4.83,6.28,14.07,6.93
4,A4 + 응급·유치원 거리,198,6.56,4.95,6.39,12.27,6.97
5,A5 공급 재고비율 → 원수치,196,6.79,5.18,6.61,13.22,6.93
6,A6 역 출입구 제외(역 좌표만),195,6.63,4.88,6.44,13.28,6.92
7,A7 역 좌표 제외(출입구만),195,6.53,4.84,6.31,14.01,6.82
8,A8 시장·전월세·금리 V1 시점(_v1),199,6.40,4.75,6.21,13.03,6.89
9,A9 15억 판정 제외,193,6.45,4.80,6.27,12.68,6.98


,구성,Feature수,Valid_MAPE_s42,Valid_MAPE_s43,Valid_MAPE_s44,Seen_MAPE,Valid신규_MAPE,신규_단지,신규_동·면적대,신규_구,홀드아웃_MAPE
0,A0 전체(196) 시드 3개,196,6.48,6.34,6.78,6.25,14.31,11.41,14.29,15.71,NaN
1,A1 예상가격 계열 제외,193,6.77,NaN,NaN,6.66,10.42,12.04,10.16,11.57,NaN
2,A12 예상가격 → 단지 이력만,196,6.37,NaN,NaN,6.26,10.00,9.77,10.18,8.82,6.77
3,A13 A12 + 주담대 계열 제외,194,6.28,NaN,NaN,6.19,9.35,10.12,8.94,11.93,6.68


| 비교 | 판단 |
|---|---|
| A0 시드 3개 | Valid 6.34~6.78% — **0.4%p 안쪽 차이는 우연 범위** |
| A1 예상가격 계열 제외 | 신규 14.3 → 10.4%로 좋아지지만 기존 단지 · 홀드아웃이 나빠짐 |
| A2 예상가격 출처 표시 추가 | 해결 안 됨 |
| A3~A9 (전세 출처 · 응급 · 유치원 · 공급 원수치 · 역 좌표/출입구 · V1 시점 · 15억) | 우연 범위 → 기존 기본값 유지 (시점이 정직한 쪽) |
| A10 · A11 permutation 0 이하 59개 제외 | 나빠짐 → 일괄 제외 안 함 |
| **A13 예상가격은 단지 자체 이력일 때만 + 대체값 기반 주담대 한도 2개 제외** | **Valid 6.28 · 신규 9.35 · 홀드아웃 6.68 — 모든 지표 개선 → 채택** |

06에 `예상가격_단지만`을 추가하고 `예상가격_만원` · `주담대_최대한도_만원` · `주담대_한도비율`을 `ablation`으로 옮겼다 (기본 Feature 194개, 5-2장 보완 후 199개).

In [4]:
LAST_TRADE = registry.loc[registry["source"].eq("06_last_trade"), "column"].tolist()
assert set(FEATURES) == set(json.load(open(EXP_DIR / "r11_features.json", encoding="utf-8"))) | set(LAST_TRADE), "기본 Feature ≠ A13 + 직전 거래"
assert "예상가격_단지만" in FEATURES and "예상가격_만원" not in FEATURES
own = trade["예상가격_출처"].eq("단지")
assert trade.loc[own, "예상가격_단지만"].equals(trade.loc[own, "예상가격_만원"]) and trade.loc[~own, "예상가격_단지만"].isna().all()
print(f"✅ 기본 Feature = A13 194개 + 직전 거래 {len(LAST_TRADE)}개 (5-2장), 예상가격_단지만 = 단지 이력일 때만 예상가격")

✅ 기본 Feature = A13 194개 + 직전 거래 7개 (5-2장), 예상가격_단지만 = 단지 이력일 때만 예상가격


## 3. 하이퍼파라미터 탐색 (보완 전 194개)

Optuna TPE, 학습률 0.1 고정, **시간 fold 2개 평균 MAPE**를 최소화한다 (`r11_tune.py`).

- fold 1: 2024-12까지 학습 → 2025H1 평가 / fold 2: 2025-06까지 학습 → 2025H2(Valid) 평가
- fold 1이 중위값보다 나쁘면 중간 종료 (MedianPruner). XGBoost는 1단계 기본 설정을 첫 시도로 넣었다.

In [5]:
con = sqlite3.connect(EXP_DIR / "r11_optuna.db")
trials = pd.read_sql(
    "select s.study_name, t.number, t.state, v.value from trials t join studies s using(study_id) "
    "left join trial_values v using(trial_id)", con)
attrs = pd.read_sql("select t.number, s.study_name, a.key, a.value_json from trial_user_attributes a "
                    "join trials t using(trial_id) join studies s using(study_id)", con)
con.close()
summary = trials.groupby("study_name").agg(시도=("number", "count"), 완료=("state", lambda s: (s == "COMPLETE").sum()),
                                           최적=("value", "min"))
best_rows = trials.loc[trials.groupby("study_name")["value"].idxmin()]
folds = attrs[attrs["key"].eq("fold_mape")].merge(best_rows, on=["study_name", "number"])
summary["최적 fold1 · fold2"] = folds.set_index("study_name")["value_json"].map(lambda v: [round(x, 2) for x in json.loads(v)])
first_xgb = trials[(trials.study_name == "r11_xgb") & (trials.number == 0)]["value"].iloc[0]
print("XGBoost 1단계 기본 설정(첫 시도):", round(first_xgb, 3))
best_params = json.load(open(EXP_DIR / "r11_best_params.json"))
print(json.dumps(best_params, indent=1, ensure_ascii=False))
summary.round(3)

XGBoost 1단계 기본 설정(첫 시도): 6.59
{
 "xgb": {
  "max_depth": 12,
  "min_child_weight": 31.742177483769634,
  "subsample": 0.6662357438035773,
  "colsample_bytree": 0.9641507700604032,
  "reg_lambda": 0.08221832913402648,
  "reg_alpha": 0.19721690639475783,
  "max_cat_to_onehot": 15
 },
 "lgb": {
  "num_leaves": 451,
  "min_child_samples": 12,
  "colsample_bytree": 0.7268667382517547,
  "subsample": 0.9648682390881465,
  "reg_lambda": 0.03963700922832873,
  "reg_alpha": 0.45659408478817315,
  "cat_smooth": 65.52911519057434,
  "max_bin": 511
 },
 "cat": {
  "depth": 8,
  "l2_leaf_reg": 32.74127435518369,
  "random_strength": 4.33744643345235,
  "bagging_temperature": 0.3834415188257777,
  "rsm": 0.8542075266578653,
  "one_hot_max_size": 8
 }
}


,시도,완료,최적,최적 fold1 · fold2
study_name,,,,
r11_cat,20,13,6.565,"[6.45, 6.68]"
r11_lgb,30,13,6.338,"[6.33, 6.35]"
r11_lgb_lasttrade,30,14,5.476,"[5.58, 5.37]"
r11_xgb,30,14,6.433,"[6.44, 6.42]"


- LightGBM이 가장 좋다 (6.34%). XGBoost는 기본 설정 대비 0.16%p 개선으로 흔들림 폭 안이다.
- CatBoost는 한 번 학습에 최대 50분이 걸리고 성능도 가장 낮다.

## 4. 최종 학습률 재학습과 앙상블 (보완 전 194개)

최적 설정을 최종 학습률(XGB · LGB 0.03, CatBoost 0.1)로 fold 2개 + 단지 홀드아웃 3개에 다시 학습했다 (`r11_refit.py`).
앙상블 후보: LightGBM 시드 3개 배깅, 정답 다양화(`log(평당가)` LightGBM), 균등 · 가중 블렌딩 (`r11_blend.py`).
가중 블렌딩 가중치는 **다른 fold에서 학습해 교차 적용**하고, 홀드아웃은 Valid를 보지 않은 fold 1 가중치로 평가했다.

**채택 기준:** 최고 단일 모델보다 두 fold 모두 0.1%p 이상 좋아지고 홀드아웃이 나빠지지 않을 때만.

In [6]:
refit = pd.read_json(EXP_DIR / "r11_refit_log.jsonl", lines=True)
display(refit.pivot(index="job", columns="split", values="mape").round(3))
blend = pd.read_csv(EXP_DIR / "r11_blend.csv")
blend.head(15)

split,f1,f2,h1,h2,h3
job,,,,,
cat_s42,7.382,7.088,7.082,7.518,7.147
lgb_py_s42,6.880,6.455,7.155,6.972,6.991
lgb_s42,6.486,6.160,6.598,6.606,6.667
lgb_s43,6.521,6.256,6.653,6.570,6.607
lgb_s44,6.588,6.189,6.650,6.623,6.631
xgb_s42,6.550,6.314,6.616,6.640,6.637


,구성,fold1,fold2(Valid),fold 평균,Valid 기존 단지,Valid 신규 단지,Valid 10% 이내,홀드아웃,비고
0,lgb_s42,6.486,6.160,6.323,6.003,11.584,82.236,6.624,NaN
1,가중 lgb_bag3+xgb_s42,6.494,6.170,6.332,6.003,11.928,82.213,6.546,"w(f1)=[0.67, 0.33] w(f2)=[0.88, 0.12]"
2,가중 lgb_bag3+xgb_s42+cat_s42,6.494,6.169,6.332,6.003,11.927,82.213,6.546,"w(f1)=[0.67, 0.33, 0.0] w(f2)=[0.88, 0.12, 0.0]"
3,가중 lgb_bag3+cat_s42,6.503,6.163,6.333,6.004,11.677,82.139,6.583,"w(f1)=[1.0, 0.0] w(f2)=[1.0, 0.0]"
4,lgb_bag3,6.503,6.163,6.333,6.004,11.677,82.139,6.583,NaN
5,균등 lgb_bag3+xgb_s42,6.491,6.189,6.340,6.019,12.064,82.112,6.548,NaN
6,가중 lgb_bag3+xgb_s42+lgb_py_s42,6.530,6.170,6.350,6.003,11.928,82.213,6.546,"w(f1)=[0.67, 0.33, 0.0] w(f2)=[0.7, 0.11, 0.19]"
7,가중 lgb_bag3+lgb_py_s42,6.538,6.163,6.350,6.004,11.677,82.139,6.583,"w(f1)=[1.0, 0.0] w(f2)=[0.81, 0.19]"
8,가중 lgb_bag3+cat_s42+lgb_py_s42,6.538,6.163,6.350,6.004,11.677,82.139,6.583,"w(f1)=[1.0, 0.0, 0.0] w(f2)=[0.81, 0.0, 0.19]"
9,가중 lgb_bag3+xgb_s42+cat_s42+lgb_py_s42,6.530,6.169,6.350,6.003,11.928,82.213,6.546,"w(f1)=[0.67, 0.33, 0.0, 0.0] w(f2)=[0.7, 0.11,..."


- 단일 LightGBM(시드 42)보다 두 fold 모두 0.1%p 이상 나은 조합이 없다. 시드 배깅은 단일 시드 기대값보다 0.03~0.04%p 낫지만 기준 미달.
- 4개 균등 블렌드는 Valid 신규 단지를 11.7 → 9.9%로 줄이지만 전체는 나빠진다.
  신규 단지만 균등 블렌드로 보내는 분기(`r11_route.py`)는 fold 2에서만 좋아지고 fold 1에서 나빠져 (11.28 → 11.47) 쓰지 않는다.
- CatBoost는 같은 설정으로 다시 학습해도 탐색 때보다 크게 나빴다 (fold 평균 6.56 → 7.24). 재현이 불안정해 후보에서 뺀다.
- **결정: LightGBM 단일 모델.** (설정은 5-2장 보완 후 다시 탐색)

## 5. 평가 방식 — 고정 분할 vs 매달 재학습

고정 분할 Test에서 MAPE가 Valid보다 크게 나빠졌다 (6.16 → 9.64%). 모든 달에서 예측이 실제보다 평균 6% 낮다.

| 진단 | 결과 |
|---|---|
| 같은 단지 실제가 / 30~395일 전 중위가 | Valid 1.056 → Test 1.074 — 2026년 상승 속도가 더 빠름 |
| 과소추정 큰 구 | 성동 -14.6% · 동작 -13.7% · 송파 -10.1% · 광진 -10.0% (2026 상승 지역) |

시장 수준 대비 정답(구 시세 · 단지 최근가 대비 비율)과 Train+Valid 재학습을 비교했다 (`r11_drift.py`).
같은 고정 분할 · 같은 빠른 설정으로 V1 Feature와도 비교했다 (`r11_v1cmp.py`, V1 노트북의 9.37%는 기준월 당월 정보가 섞인 값이라 직접 비교하지 않는다).

In [7]:
display(pd.read_csv(EXP_DIR / "r11_drift.csv").round(2))
pd.read_csv(EXP_DIR / "r11_v1cmp.csv")

,구성,Valid_MAPE,Valid_편향,Test_MAPE,Test_편향
0,abs | Train 학습,6.16,-0.60,9.64,-6.02
1,abs | Train+Valid 재학습 (3132회),NaN,NaN,8.05,-4.65
2,구평당가 | Train 학습,7.34,-2.35,11.43,-7.88
3,구평당가 | Train+Valid 재학습 (2386회),NaN,NaN,9.26,-6.14
4,단지예상가 | Train 학습,6.20,-1.12,9.41,-5.04
5,단지예상가 | Train+Valid 재학습 (3119회),NaN,NaN,8.16,-3.91


,구성,Feature수,Valid_MAPE,Test_MAPE,Test_편향
0,V1 시점 그대로,56,9.25,13.26,-8.78
1,V1 정의 + V2 시점,60,8.64,12.46,-7.72
2,V2 기본(A13),194,6.48,9.92,-5.65


- 정답 변환은 편향을 조금 줄이지만 MAPE가 나아지지 않는다 → **정답은 `log(거래금액)` 유지.**
- 같은 조건에서 **V2가 V1보다 Test 3.3%p 낫다** (13.26 → 9.92%, 학습률 0.1 기준).
- 서비스는 매달 재학습하므로, 서비스 성능은 **매달 재학습 평가**로 잰다:
  m월 예측 = m월 1일에 확정된 거래(≤ m-2월)로 학습한 모델. 2022-01~2026-06 54개월 모두 이 방식으로 예측했다 (OOF).
  보완 전 모델(`r11_rolling.py`): Valid 6.06% · Test 6.77% (중개 5.70 · 6.52%).

### 최근 확정월 보정 — 기각 (보완 전 모델로 확인)

m-2월 OOF 잔차(서울 또는 구 중위)를 m월 예측에 더하는 보정은 Test에서 6.77 → 6.36%로 좋아졌지만,
**Valid에서는 6.06 → 6.59%로 나빠졌다.** Test를 보고 고르는 셈이 되므로 쓰지 않는다.

In [8]:
rolling = pd.read_csv(EXP_DIR / "r11_rolling.csv")
rolling["구간"] = np.select([rolling["월"].between("2025-07", "2025-12"), rolling["월"] >= "2026-01"], ["Valid", "Test"], "")
rolling = rolling[rolling["구간"] != ""]
rolling.groupby("구간")[["n", "MAPE", "MAPE_서울보정", "MAPE_구보정"]].apply(
    lambda g: pd.Series({c: np.average(g[c], weights=g["n"]) for c in ["MAPE", "MAPE_서울보정", "MAPE_구보정"]})).round(3)

,MAPE,MAPE_서울보정,MAPE_구보정
구간,,,
Test,6.773,6.355,6.412
Valid,6.058,6.585,6.506


## 5-2. 보완 — 같은 단지 · 같은 면적 직전 거래

보완 전 모델을 **사용자가 이미 보는 값**(같은 단지 · 같은 평형 최근 실거래가)과 비교하니 거의 같았다
(Test 중개거래, 직전 거래 있는 거래: MAPE 6.36 vs 6.92%, 중앙 5.23 vs 5.26%). 모델은 단지 전체 1년 중위값(`예상가격_단지만`)만 알고
같은 면적의 가장 최근 거래라는 가장 강한 신호를 몰랐다.

- 06에 Feature 5개 추가 (`06_last_trade`): 직전 거래가 · 직전 3건 중위 · 경과일 · 직전 거래 층 · 같은 면적 누적 거래 수.
  계약일 30일 전까지 계약된 중개거래만 (신고 기한 30일, `예상가격`과 같은 규칙), 경계일 골든 테스트 포함.
- 빠른 확인 (Train → Valid, 같은 설정): 아래 출력. 시드 흔들림(0.4%p)의 2배가 넘는 개선이라 채택.
- LightGBM 재탐색 30회 (`r11_tune2.py`, 보완 전 최적 설정을 첫 시도로): fold 평균 5.52 → 5.48%로 차이는 작지만,
  잎 수가 451 → 72로 줄어 모델이 작고 빠르다 → 새 설정 채택. 반복 수는 Train → Valid 조기 종료 × 1.15.
- 같은 방식으로 54개월 매달 재학습을 다시 했다 (`r11_rolling2.py`).

In [9]:
print(open(EXP_DIR / "lt_test_output.txt", encoding="utf-8").read().replace("  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)\n", ""))
con = sqlite3.connect(EXP_DIR / "r11_optuna.db")
lt = pd.read_sql("select t.number, t.state, v.value from trials t join studies s using(study_id) left join trial_values v using(trial_id) "
                 "where s.study_name = 'r11_lgb_lasttrade'", con)
con.close()
final_params = json.load(open(EXP_DIR / "r11_best_params_lasttrade.json", encoding="utf-8"))
print("재탐색:", len(lt), "회 | 첫 시도(보완 전 설정)", round(lt.loc[lt.number == 0, "value"].iloc[0], 3), "| 최적", round(lt["value"].min(), 3))
print(json.dumps(final_params, indent=1, ensure_ascii=False))
compare = pd.read_csv(EXP_DIR / "r11_lasttrade_compare.csv")
compare

기존 194 iter 2724 Valid 6.16 중개 5.799 중개 중앙 4.365 신규 11.58 769 초
+ 직전 거래 5개 iter 857 Valid 5.242 중개 4.828 중개 중앙 3.491 신규 11.8 1489 초

[exited with code 0]

재탐색: 30 회 | 첫 시도(보완 전 설정) 5.521 | 최적 5.476
{
 "lgb": {
  "num_leaves": 72,
  "min_child_samples": 24,
  "colsample_bytree": 0.4322311163924917,
  "subsample": 0.9944803483540686,
  "reg_lambda": 46.27272675640742,
  "reg_alpha": 0.010095396757182767,
  "cat_smooth": 16.406899587916314,
  "max_bin": 255
 },
 "n_estimators": 4290,
 "learning_rate": 0.03,
 "train_valid_mape": 5.329470182178547,
 "best_iteration": 3734
}


,방법,구간,직전 거래 있는 거래 MAPE,직전 거래 있는 거래 중앙 APE,직전 거래 있는 거래 10%이내,전체 MAPE,직전 거래 없는 거래 MAPE,직전 거래 있음 비율
0,보완 후 모델,Valid,4.840,3.594,89.484,4.947,10.733,98.186
1,보완 후 모델,Test,5.374,4.157,87.433,5.525,12.091,97.738
2,보완 전 모델,Valid,5.609,4.291,84.612,5.696,10.383,98.186
3,보완 전 모델,Test,6.408,5.255,80.108,6.523,11.483,97.738
4,같은 단지·면적 직전 거래가,Valid,6.312,4.564,80.923,NaN,NaN,98.186
5,같은 단지·면적 직전 거래가,Test,6.665,4.839,79.317,NaN,NaN,97.738
6,같은 단지·면적 직전 3건 중위,Valid,6.490,4.762,79.911,NaN,NaN,98.186
7,같은 단지·면적 직전 3건 중위,Test,6.926,5.114,77.726,NaN,NaN,97.738


- **보완 후 모델이 직전 거래가를 이긴다:** Test 직전 거래 있는 거래 MAPE 5.37 vs 6.67%, 중앙 4.16 vs 4.84%, 10% 이내 87.4 vs 79.3%.
- 보완 전 모델보다 Test 1.0%p 낫다 (6.41 → 5.37%). 직전 거래가 없는 거래(2.3%)는 그대로 10~12%다.

## 5-3. 보완 2 — 직전 거래가 시세 보정

직전 거래가는 **그 거래 당시** 가격이라, 그 뒤 시장이 움직인 만큼 어긋난다. 5-2 모델의 한계였던 2022 하반기 급락장(오차 12~16%)이 이 경우다.
데이터를 더 모아 후보 4묶음을 같은 규칙으로 비교했다.

| 후보 | 출처 | Feature |
|---|---|---|
| 상권 규모 | 소상공인 상가정보 (02c와 같은 스냅샷 시점 규칙) | 단지 주변 점포 · 카페 · 음식점 수 등 7개 |
| 배정 초등학교 | 학구도(한국교육시설안전원) + EDSS 학교 통계 | 배정 학교 거리 · 학생 수 · 학급당 학생 수 등 8개 (거래 23%는 배정 학교 ≠ 최근접 학교) |
| 단지 속성 | K-apt 공동주택 기본 · 상세 (서울 3,415단지, 거래 85% 연결) | 복도유형 · 난방 · 주상복합 · 분양형태 · 시공사 · 지하주차 비중 7개 |
| 시세 보정 | 부동산원 구별 주간 아파트 매매가격지수 | 직전 거래 뒤 구 지수 변화 · 직전 거래가 × 지수 비율 2개 |

- **평가 잡음 발견 (`r11_noise_check.py`):** 처음 점검은 학습률 0.06 + 평가 구간 조기 종료라 트리 수백 개에서 멈췄다.
  시드만 바꿔도 홀드아웃이 최대 0.9%p 흔들리고, **의미 없는 난수 열 7개를 더해도 비슷한 크기로 "좋아졌다"** (아래 첫 표). → 이 방식의 판정은 모두 버렸다.
- **본 모델 설정으로 재비교 (`r11_eval_fixed.py`):** 학습률 0.03, 트리 수 고정(5-2 튜닝 best_iteration 3,734), 평가 구간 조기 종료 없음, 시드 2개.
  시드 간 차이가 채택 기준(0.1%p)보다 작은지 먼저 확인한다.
- 채택 규칙은 그대로: 두 fold(f1 · f2) 모두 0.1%p 이상 개선, 단지 홀드아웃(h1~h3 평균) 악화 없음. 모두 중개거래 MAPE.


In [10]:
noise = pd.read_csv(EXP_DIR / "r11_noise_check.csv")
print("점검 방식(학습률 0.06 + 평가 구간 조기 종료) — 시드별 MAPE")
display(noise.pivot_table(index=["구성", "시드"], columns="평가", values="MAPE").round(2))

fixed = pd.read_csv(EXP_DIR / "r11_eval_fixed.csv")
fixed["평가"] = fixed["평가"].replace({"h1": "홀드아웃", "h2": "홀드아웃", "h3": "홀드아웃"})
order = fixed["구성"].unique()
summary = fixed.pivot_table(index="구성", columns="평가", values="MAPE").reindex(order)
summary["10%이내 f1"] = fixed[fixed["평가"].eq("f1")].groupby("구성")["10%이내"].mean()
summary["시드 간 최대 차이"] = fixed.groupby(["평가", "구성", "시드"])["MAPE"].mean().unstack().pipe(lambda d: d.max(axis=1) - d.min(axis=1)).groupby("구성").max()
base = summary.loc["현재 11", ["f1", "f2", "홀드아웃"]]
summary["채택"] = [(r["f1"] <= base["f1"] - 0.1) and (r["f2"] <= base["f2"] - 0.1) and (r["홀드아웃"] <= base["홀드아웃"]) for _, r in summary.iterrows()]
print("본 모델 설정 재비교 — 시드 2개 평균 MAPE")
summary.round(3)


점검 방식(학습률 0.06 + 평가 구간 조기 종료) — 시드별 MAPE


평가            f1  홀드아웃 h1  홀드아웃 h2  홀드아웃 h3
구성      시드                                 
+ 난수 7개 1    NaN     5.35     5.33     5.15
        42  5.80      NaN      NaN      NaN
        43  6.30      NaN      NaN      NaN
현재 11   1    NaN     6.01     5.80     5.84
        2    NaN     5.23     5.28     5.07
        3    NaN     5.16     5.21     4.89
        44  5.47      NaN      NaN      NaN
        45  5.88      NaN      NaN      NaN
        46  5.59      NaN      NaN      NaN

본 모델 설정 재비교 — 시드 2개 평균 MAPE


평가,f1,f2,홀드아웃,10%이내 f1,시드 간 최대 차이,채택
구성,,,,,,
현재 11,5.599,4.989,5.126,86.517,0.283,False
+ 상권 7개,5.886,4.972,5.139,84.905,0.167,False
+ 배정초 8개,5.810,5.007,5.173,85.345,0.045,False
+ K-apt 7개,5.854,4.934,5.155,84.938,0.153,False
+ 시세보정 2개,5.312,4.844,4.956,88.276,0.028,True


- 잡음 점검: 같은 설정에서 시드만 바꿔도 h1이 6.01 / 5.23 / 5.16%로 흔들렸고, 난수 열(5.35%)이 "개선"처럼 보였다. 평가 구간을 보고 멈추는 것 자체가 낙관 편향이다.
- 본 모델 설정에서는 시드 간 차이가 대부분 0.1%p 안쪽이라 판정할 수 있다.
- **상권 · 배정 초등학교 · K-apt는 불채택**: f1에서 오히려 나빠졌다. 공시가격과 직전 거래가가 단지의 고정 속성(입지 · 학군 · 구조)을 이미 담고 있다.
  → 모델 Feature가 아니라 **웹 화면 표시값**으로 쓴다 (배정 초등학교, 복도유형 · 난방 · 시공사 · 지하주차, 주변 카페 · 음식점 수).
- **시세 보정 2개 채택**: f1 5.60 → 5.31, f2 4.99 → 4.84, 홀드아웃 5.13 → 4.96%. 모델이 몰랐던 것은 고정 속성이 아니라 **직전 거래 뒤의 시세 변화**였다.
- 06에 반영 (`06_last_trade`, 기본 Feature 201개):
  - `직전거래후_구지수변화` = log(지금 지수 / 직전 거래 주 지수), `직전거래가_시세보정` = 직전 거래가 × 지수 비율
  - 지금 지수는 **계약일 전에 공표된 주만** 쓴다 (주간 지수는 기준일 + 3일 공표). 직전 거래 주 지수는 그 주 값
  - 지수가 시작되는 2021-07-05 이전 직전 거래는 결측(12%, 대부분 Train 초기). 공표 시점 · 잘라내기 검증 통과
- 반복 수를 5-2와 같은 방식으로 다시 정하고(Train → Valid 조기 종료 1,805 × 1.15 = 2,080회) 54개월 매달 재학습을 다시 했다 (`r11_rolling3.py`).
  아래는 같은 거래(중개)에 대한 v2.2 OOF(5-2 모델)와의 비교, 같은 단지 · 면적 직전 거래 기준선, 반기별 MAPE다.


In [11]:
final_params = json.load(open(EXP_DIR / "r11_best_params_idx.json", encoding="utf-8"))
print(json.dumps(final_params, indent=1, ensure_ascii=False))
compare = pd.read_csv(EXP_DIR / "r11_idx_compare.csv")
display(compare)
pd.read_csv(EXP_DIR / "r11_idx_by_half.csv", index_col=0)


{
 "lgb": {
  "num_leaves": 72,
  "min_child_samples": 24,
  "colsample_bytree": 0.4322311163924917,
  "subsample": 0.9944803483540686,
  "reg_lambda": 46.27272675640742,
  "reg_alpha": 0.010095396757182767,
  "cat_smooth": 16.406899587916314,
  "max_bin": 255
 },
 "n_estimators": 2080,
 "learning_rate": 0.03,
 "train_valid_mape": 5.213433416740411,
 "best_iteration": 1805
}


,방법,구간,직전 거래 있는 거래 MAPE,직전 거래 있는 거래 중앙 APE,직전 거래 있는 거래 10%이내,전체 MAPE,직전 거래 없는 거래 MAPE,직전 거래 있음 비율
0,보완 2 모델 (시세 보정),Valid,4.748,3.511,90.066,4.857,10.777,98.186
1,보완 2 모델 (시세 보정),Test,5.192,3.891,88.202,5.348,12.087,97.738
2,v2.2 모델,Valid,4.840,3.594,89.484,4.947,10.733,98.186
3,v2.2 모델,Test,5.374,4.157,87.433,5.525,12.091,97.738
4,직전 거래가 × 시세 보정,Valid,5.684,4.105,83.204,NaN,NaN,98.186
5,직전 거래가 × 시세 보정,Test,6.228,4.467,80.844,NaN,NaN,97.738
6,같은 단지·면적 직전 거래가,Valid,6.312,4.564,80.923,NaN,NaN,98.186
7,같은 단지·면적 직전 거래가,Test,6.665,4.839,79.317,NaN,NaN,97.738
8,같은 단지·면적 직전 3건 중위,Valid,6.490,4.762,79.911,NaN,NaN,98.186
9,같은 단지·면적 직전 3건 중위,Test,6.926,5.114,77.726,NaN,NaN,97.738


,v2.2 MAPE,보완 2 MAPE
반기,,
2022H1,6.77,6.78
2022H2,12.92,12.46
2023H1,6.11,5.91
2023H2,6.06,5.87
2024H1,4.28,4.26
2024H2,4.71,4.47
2025H1,4.81,4.61
2025H2,4.95,4.86
2026H1,5.53,5.35


- **매달 재학습 Test 중개 MAPE 5.53 → 5.35%, Valid 4.95 → 4.86%** (그 이전 2022-01~2025-06도 5.28 → 5.11%). 2022 상반기를 빼고 모든 반기에서 낫다.
- 직전 거래 있는 Test 거래: **모델 5.19%** vs 직전 거래가 × 시세 보정 6.23% vs 직전 거래가 6.67%. 보정값만 써도 나아지지만 모델이 1%p 더 정확하다.
- 2026 상반기 과소추정 편향이 -2.10 → -1.51%로 줄었다.
- 2022 하반기 급락장은 12.92 → 12.46%로 조금 나아졌을 뿐이다. 구 지수는 구 평균이고, 거래 절벽에서 단지별 낙폭은 제각각이다 → 한계로 남긴다.
- 잎 72 등 LightGBM 설정은 5-2 재탐색 그대로, 반복 수는 4,290 → 2,080회 (강한 Feature가 생겨 더 일찍 수렴).
- **이후 6~11장은 이 OOF(`oof_predictions.parquet` = `r11_oof3.parquet`)와 이 설정으로 다시 계산한다.**


In [12]:
oof = pd.read_parquet(MODEL_DIR / "oof_predictions.parquet")
oof["월"] = oof["월"].astype("period[M]")
oof["bias"] = (np.exp(oof["p"] - oof["y"]) - 1) * 100
VALID_M = pd.period_range("2025-07", "2025-12", freq="M")
TEST_M = pd.period_range("2026-01", "2026-06", freq="M")
oof["구간"] = np.select([oof["월"].isin(VALID_M), oof["월"].isin(TEST_M)], ["Valid", "Test"], "그 이전")
MARKET = oof["거래유형"].eq("중개거래")

perf = oof[oof["구간"] != "그 이전"].groupby(["구간", "거래유형"]).agg(n=("ape", "size"), MAPE=("ape", "mean"),
                                                                     중앙APE=("ape", "median"), 편향=("bias", "mean"))
perf["10%이내"] = oof[oof["구간"] != "그 이전"].groupby(["구간", "거래유형"])["ape"].apply(lambda s: (s <= 10).mean() * 100)
overall = oof[oof["구간"] != "그 이전"].groupby("구간").agg(n=("ape", "size"), MAPE=("ape", "mean"), 편향=("bias", "mean"))
display(overall.round(2))
display(perf.round(2))
print("연도별 중개거래 MAPE:", oof[MARKET].groupby(oof["월"].dt.year)["ape"].mean().round(2).to_dict())
oof[oof["구간"] != "그 이전"].groupby("월").agg(MAPE=("ape", "mean"), 편향=("bias", "mean")).round(2).T

,n,MAPE,편향
구간,,,
Test,39569,5.65,-1.30
Valid,33788,5.25,-0.94


n   MAPE  중앙APE    편향  10%이내
구간    거래유형                                  
Test  중개거래  37923   5.35   3.95 -1.51  87.39
      직거래    1646  12.59   8.89  3.54  52.92
Valid 중개거래  32193   4.86   3.55 -1.26  89.49
      직거래    1595  13.27  10.42  5.52  47.21

연도별 중개거래 MAPE: {2022: 8.65, 2023: 5.89, 2024: 4.37, 2025: 4.72, 2026: 5.35}


월,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06
MAPE,5.37,5.15,4.70,5.75,5.46,5.22,4.95,5.89,5.86,5.80,5.85,5.28
편향,0.35,1.43,-1.31,-2.48,-1.52,-0.34,-1.41,-2.72,-1.97,-0.44,-0.30,-2.01


## 5-4. 보완 3 — 비교 거래가 드문 거래 (불채택)

**남은 여유부터 확인** (`r11_noise_floor.py`, 중개거래). 같은 단지 · 면적 · 같은 달 다른 거래의 실제 가격 평균은 예측 시점에 알 수 없는 값이다. 비교 거래도 한두 건이라 그 자체 오차가 섞여 엄밀한 하한은 아니다.

| 구분 | 2023~25 MAPE / 중앙 | Test MAPE / 중앙 |
|---|---:|---:|
| 비교 거래 있음: 같은 달 다른 거래 평균 | 3.85 / 2.77 | 4.47 / 3.16 |
| 비교 거래 있음: 모델 | 4.29 / 3.33 | 4.86 / 3.70 |
| 비교 거래 없음 · 기존 단지: 모델 | 5.47 / 4.01 | 6.03 / 4.36 |
| 신규 단지: 모델 | 11.23 / 8.10 | 13.61 / 9.58 |

**후보** (`r11_nb_build.py`, 참고 거래는 계약일 30일 전보다 앞선 직거래 외 거래, 구 주간 지수로 보정). 이 구간을 살펴본 계기는 위 Test 표이고, 후보 선택과 채택 판정에는 Test를 쓰지 않았다.

- 단지 3개: 같은 단지 모든 면적 거래의 시세 보정 평당가 중위수 × 면적, 창 안 거래 수, 최근 거래 경과일 (`예상가격_단지만`은 시세 보정이 없는 12개월 중위수)
- 유사 2개: 같은 구에서 좌표 · 준공연도 · 세대수가 가까운 10개 단지의 시세 보정가, 창 안 거래 수. 21의 유사 단지 목록은 2026-07 속성으로 이웃을 골라 과거 거래에 쓰면 미래 속성이 섞이고 준공연도가 없어 쓰지 않았다

**결과** (`r11_nb_test.csv`, 본 모델 설정 × 시드 2개 평균, 중개 MAPE)

| 구성 | f1 | f2 | 홀드아웃 3개 평균 | 판정 |
|---|---:|---:|---:|---|
| 현재 11 | 5.259 | 4.847 | 4.929 | 기준 |
| + 단지 3개 | 5.312 | 4.785 | 4.869 | f1 악화 |
| + 유사 2개 | 5.229 | 4.847 | 4.946 | 시드 간 f1 차이 0.28%p, 시드 42는 전 구간 악화 |
| + 단지 · 유사 5개 | 5.320 | 4.775 | 4.853 | f1 악화, f2 개선 0.07%p |

- **채택 없음.** 채택 규칙(두 시드 모두 f1 · f2에서 0.1%p 이상 개선, 홀드아웃 악화 없음)을 만족한 구성이 없다. 규칙을 결과에 맞춰 낮추지 않는다.
- 노린 구간도 뚜렷이 좋아지지 않았다: 5개 추가 시 직전 거래 6개월+ · 없음 f1 7.11 → 7.07%, f2 7.73 → 7.69%, 100세대 미만 f1 7.75 → 7.66%, f2 7.94 → 7.89%.

**추가 점검 — 층 관계 Feature** (`r11_floor_test.csv`, 같은 설정 · 규칙)

| 구성 | f1 | f2 | 홀드아웃 3개 평균 | 판정 |
|---|---:|---:|---:|---|
| 현재 11 | 5.259 | 4.847 | 4.929 | 기준 |
| + 층 차이 1개 (`층 - 직전거래_층`) | 5.547 | 4.860 | 4.928 | f1 악화 |
| + 층 3개 (층 차이 · 최고층 대비 상대 층 · 2층 이하) | 5.473 | 4.799 | 4.887 | f1 악화, f2 개선 0.05%p |

채택 없음. 이미 있는 층 · 직전거래_층 · 최고층의 계산값이라 새 정보가 없다. 호 단위 공시가격처럼 새 정보를 넣는 후보는 이 결과로 판단할 수 없다.

**남은 여유의 해석** (`r11_noise_floor_adj.csv`, 2023~25, Test 제외). 위 표의 기준에는 비교 거래 자체의 잡음이 섞여 있어, 비교 건수로 잡음을 뺀 추정을 함께 본다.

| 비교 기준 | 기준 오차 | 잡음 제거 추정 | 같은 행의 모델 |
|---|---:|---:|---:|
| 같은 달 다른 거래 평균 | 3.85 | 3.11 | 4.29 |
| 층 차이 2 이내 거래만 | 3.45 | 2.59 | 4.29 |

- 모델과의 격차는 0.4%p가 아니라 추정 1.2~1.7%p다. 이 기준은 같은 달 시세를 아는 값이라 격차에는 줄일 수 없는 정보 지연이 포함되고, 그 비율은 측정하지 못했다.
- f1은 Feature를 하나만 더해도 0.2~0.4%p 움직여(기준 시드 간 차이는 0.02%p) 작은 개선을 판정하기 어렵다. 판정 규칙은 그대로 두되 이 한계를 기록한다.
- 신규 단지는 지금처럼 등급 없이 참고용 범위로 둔다.


## 6. 오류가 큰 곳 (Test, 중개거래)

In [13]:
test_oof = oof[oof["구간"].eq("Test") & MARKET].merge(trade[["transaction_id", "시군구명", "연식"]], on="transaction_id")
display(test_oof.groupby(["단지", "가격대"]).agg(n=("ape", "size"), MAPE=("ape", "mean"), 편향=("bias", "mean")).round(2))
display(test_oof.groupby("시군구명").agg(n=("ape", "size"), MAPE=("ape", "mean"), 편향=("bias", "mean")).round(2)
        .sort_values("편향").head(6))
test_oof.groupby(pd.cut(test_oof["연식"], [-1, 1, 5, 15, 30, 100]), observed=True).agg(n=("ape", "size"), MAPE=("ape", "mean"), 편향=("bias", "mean")).round(2)

n   MAPE    편향
단지 가격대                       
기존 15억 이상   8405   4.96  0.14
   6~9억    10342   5.08 -2.11
   6억 미만    7793   5.49 -0.95
   9~15억   11051   5.54 -2.67
신규 15억 이상     43  13.12 -0.59
   6~9억      108  12.93  2.42
   6억 미만     120  12.31  0.49
   9~15억      61  17.72  1.36

,n,MAPE,편향
시군구명,,,
동작구,1308,5.53,-2.80
동대문구,1639,5.51,-2.70
강북구,896,4.61,-2.37
성북구,2344,4.99,-2.27
광진구,507,6.50,-2.15
서대문구,1503,5.34,-2.08


,n,MAPE,편향
연식,,,
"(-1, 1]",186,9.27,-1.51
"(1, 5]",1964,4.95,-0.58
"(5, 15]",6028,4.71,-0.70
"(15, 30]",19346,5.39,-1.64
"(30, 100]",10399,5.65,-1.91


- 매달 재학습해도 과소추정이 남는다 (Test 중개 평균 -1.5%, 동작 -2.8% · 동대문 -2.7% · 강북 -2.4%). 확정 거래가 2개월 늦게 들어오는 구조적 한계다.
  보완 전 모델(-3.2%)보다, 5-2 모델(-2.1%)보다 줄었다 — 직전 거래가가 30일 전까지의 거래를, 시세 보정이 계약일 직전 주까지의 지수를 반영하기 때문이다.
  가격대로는 6~15억(-2.1 ~ -2.7%), 연식으로는 15년 이상 구축(-1.6 ~ -1.9%)에서 편향이 크다.
- 신규 단지(학습 시점에 확정 거래가 없는 단지)는 오차가 크다 (12~18%) → 7장에서 따로 처리.


## 7. 80% 오차 범위

잔차 `r = log(실제) - log(예측)`의 분위수로 구간을 만든다. 구간은 OOF 잔차로만 만든다 (학습 거래 잔차는 오차를 과소평가).

- **세그먼트:** 단지(기존/신규) × 예측가 가격대(6억 미만 · 6~9억 · 9~15억 · 15억 이상)
- **이동 창:** m월 구간 = m-(창+1) ~ m-2월 중개거래 잔차 (m월 1일에 알 수 있는 값)
- **창 길이: 3개월 고정.** 12개월 창은 추세가 있는 시장에서 오래된 잔차를 써서 포함률이 계속 떨어진다
  (Test 이전 반기별 표: 8~92% 분위수 기준 2024H1 91% → 2025H1 83% → 2025H2 80%). 3개월 창은 같은 기간 80~87%로 안정적이다.
- **분위수:** 3개월 창에서 0.10~0.05를 **Valid에서** 비교해 80% 이상 중 폭이 가장 좁은 것 → Test는 확인만
- 기록: 처음에는 창 길이도 Valid에서 골랐다(80% 이상 중 최소 폭). 보완 2 모델에서는 12개월 창이 Valid 80.2%로 아슬아슬하게 뽑혔고 Test 포함률이 78.8%였다.
  그 뒤 Test 이전 반기별 포함률을 보고 창 길이 규칙을 위처럼 바꿨다 (보완 전 모델은 원래 규칙으로도 3개월 창이 뽑혔다).

In [14]:
SEG = ["단지", "가격대"]


def interval_table(month, alpha, window, data=oof):
    w = data[data["거래유형"].eq("중개거래") & data["월"].between(month - (window + 1), month - 2)]
    q = w.groupby(SEG)["r"].quantile([alpha, 1 - alpha]).unstack()
    q.columns = ["lo", "hi"]
    q_all = w["r"].quantile([alpha, 1 - alpha]).to_numpy()
    return q, q_all


def attach_interval(rows, month, alpha, window):
    q, q_all = interval_table(month, alpha, window)
    key = pd.MultiIndex.from_frame(rows[SEG])
    out = rows.copy()
    out["lo"] = q["lo"].reindex(key).fillna(q_all[0]).to_numpy()
    out["hi"] = q["hi"].reindex(key).fillna(q_all[1]).to_numpy()
    out["포함"] = out["r"].between(out["lo"], out["hi"])
    out["폭%"] = (np.exp(out["hi"]) - np.exp(out["lo"])) * 100
    return out


def evaluate(months, alpha, window):
    return pd.concat([attach_interval(oof[oof["월"].eq(m)], m, alpha, window) for m in months], ignore_index=True)


grid = []
for window in (3, 6, 12):
    for alpha in (0.10, 0.08, 0.07, 0.06, 0.05):
        ev = evaluate(VALID_M, alpha, window)
        mk = ev["거래유형"].eq("중개거래") & ev["단지"].eq("기존")
        grid.append({"창": window, "분위수": alpha, "Valid 포함률(기존)": ev.loc[mk, "포함"].mean() * 100,
                     "Valid 폭%(기존)": ev.loc[mk, "폭%"].mean()})
grid = pd.DataFrame(grid)

# 창 길이 안정성: Test 이전 반기별 포함률 (기존 단지 · 중개, 8~92% 분위수)
pre = pd.period_range("2023-01", "2025-12", freq="M")
stab = {}
for window in (3, 6, 12):
    ev = evaluate(pre, 0.08, window)
    mk = ev["거래유형"].eq("중개거래") & ev["단지"].eq("기존")
    half = ev["월"].dt.year.astype(str) + np.where(ev["월"].dt.month <= 6, "H1", "H2")
    stab[f"{window}개월"] = ev[mk].groupby(half[mk])["포함"].mean() * 100
display(pd.DataFrame(stab).T.round(1))

chosen = grid[grid["창"].eq(3) & (grid["Valid 포함률(기존)"] >= 80)].sort_values("Valid 폭%(기존)").iloc[0]
WINDOW, ALPHA = int(chosen["창"]), float(chosen["분위수"])
print(f"선택: 창 {WINDOW}개월, 분위수 {ALPHA:.2f}~{1 - ALPHA:.2f}")
grid.round(2)

월,2023H1,2023H2,2024H1,2024H2,2025H1,2025H2
3개월,84.0,80.2,86.9,79.6,80.0,80.9
6개월,84.9,83.6,88.5,81.2,83.6,79.4
12개월,87.2,85.8,91.2,86.5,83.4,80.2


선택: 창 3개월, 분위수 0.08~0.92


,창,분위수,Valid 포함률(기존),Valid 폭%(기존)
0,3,0.10,76.28,13.17
1,3,0.08,80.88,14.77
2,3,0.07,83.08,15.77
3,3,0.06,85.47,16.93
4,3,0.05,87.82,18.33
5,6,0.10,74.83,12.89
6,6,0.08,79.41,14.43
7,6,0.07,81.87,15.42
8,6,0.06,84.30,16.53
9,6,0.05,86.72,17.86


In [15]:
test_iv = evaluate(TEST_M, ALPHA, WINDOW)
mk = test_iv["거래유형"].eq("중개거래")
print("Test 포함률 (중개, 기존 단지):", round(test_iv.loc[mk & test_iv["단지"].eq("기존"), "포함"].mean() * 100, 1),
      "| 평균 폭:", round(test_iv.loc[mk & test_iv["단지"].eq("기존"), "폭%"].mean(), 1))
display(test_iv[mk].groupby(SEG).agg(n=("포함", "size"), 포함률=("포함", "mean"), 폭=("폭%", "mean"))
        .assign(포함률=lambda d: d["포함률"] * 100).round(1))
test_iv[mk & test_iv["단지"].eq("기존")].groupby(test_iv["월"].astype(str))["포함"].mean().mul(100).round(1).to_frame("월별 포함률(기존)").T

Test 포함률 (중개, 기존 단지): 81.8 | 평균 폭: 16.4


n   포함률     폭
단지 가격대                      
기존 15억 이상   8405  83.1  17.1
   6~9억    10342  80.1  15.0
   6억 미만    7793  81.8  17.2
   9~15억   11051  82.5  16.7
신규 15억 이상     43  74.4  48.9
   6~9억      108  72.2  43.8
   6억 미만     120  81.7  40.0
   9~15억      61  65.6  44.2

월,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06
월별 포함률(기존),84.2,80.1,78.4,80.1,82.0,87.2


- **기존 단지:** Test 포함률 81.8% (가격대별 80.1~83.1%, 월별 78.4~87.2%), 평균 폭 16.4% (5-2 모델 16.6%, 보완 전 20.4%).
- **신규 단지:** 포함률 66~82%, 폭 40~49%. 보완 전 모델로 가격대를 합쳐 분위수를 Valid에서 따로 맞춰도(`r11_interval3.py`: Valid 82.5%) Test는 72%였다.
  거래 이력이 없는 단지는 구간을 넓혀도 믿을 만한 범위가 되지 않는다.
  → **신규 단지는 등급을 매기지 않고, 범위를 "거래 이력 없음 · 참고용"으로만 보여준다.**


## 8. 적정가 대비 등급 (규칙)

`r = log(실제 또는 호가 / 적정가)`를 구간 `[lo, hi]`와 비교한다. 초과 정도는 해당 방향 구간 반폭 기준이다.

| 등급 | 판정 |
|---|---|
| 적정 | `lo ≤ r ≤ hi` |
| 다소 고평가 / 다소 저평가 | 구간 밖, 초과분이 반폭의 1배 이하 (= 적정가에서 반폭의 1~2배) |
| 고평가 / 저평가 | 초과분이 반폭의 1배 초과 (= 반폭의 2배 초과) |
| 등급 없음 (참고용) | 신규 단지 |

실제 거래가가 범위보다 높으면 "고평가", 낮으면 "저평가"다. 사용자 표현은 "모델 기준"으로 고정한다.

In [16]:
def grade(r, lo, hi, complex_kind):
    if complex_kind == "신규":
        return "등급 없음(참고용)"
    if lo <= r <= hi:
        return "적정"
    if r > hi:
        return "다소 고평가" if (r - hi) <= hi else "고평가"
    return "다소 저평가" if (lo - r) <= -lo else "저평가"


test_iv["등급"] = [grade(a, b, c, k) for a, b, c, k in zip(test_iv["r"], test_iv["lo"], test_iv["hi"], test_iv["단지"])]
ORDER = ["저평가", "다소 저평가", "적정", "다소 고평가", "고평가", "등급 없음(참고용)"]
dist = pd.crosstab(test_iv["거래유형"], test_iv["등급"], normalize="index").mul(100).reindex(columns=ORDER).round(1)
dist

등급,저평가,다소 저평가,적정,다소 고평가,고평가,등급 없음(참고용)
거래유형,,,,,,
중개거래,3.0,6.3,81.1,7.8,1.0,0.9
직거래,18.2,5.4,27.7,12.3,4.1,32.3


- 중개거래는 81.1%가 적정이고, 구간 밖은 고평가 쪽 8.8% · 저평가 쪽 9.3%로 비슷하다.
- 직거래는 저평가 비율이 크다 — 가족 간 거래 등 시세보다 싼 거래가 섞인다는 해석과 맞는다. 웹에서 거래 목록에 등급을 붙일 때 직거래는 따로 표시한다.


## 9. OOF Gap 구 × 월 (31 분류 입력)

`price_gap_pct = 실제 / 예측 - 1`을 거래월 × 구로 집계한다. 모든 예측이 OOF(그 달 시작 시점에 확정된 거래로 학습)라 회귀 과적합이 섞이지 않는다.

- 거래월 M의 Gap은 신고 기한 때문에 **M+2월 1일부터** 알 수 있다 → `사용가능월` 열로 남기고, 31은 예측시점 p에 `사용가능월 ≤ p`인 값만 쓴다.
- 2022년은 학습 데이터가 짧아(첫 달 5개월치) OOF 오차가 크다 (중개 MAPE 9.4%). 31에서 Gap Feature는 Ablation으로 채택 여부를 정한다.

In [17]:
gap = oof.merge(trade[["transaction_id", "시군구명"]], on="transaction_id")
gap["gap"] = np.exp(gap["r"]) - 1
gu_month = (gap.groupby(["시군구명", "월"])
            .agg(거래수=("gap", "size"), gap_평균=("gap", "mean"), gap_중위=("gap", "median"))
            .join(gap[gap["거래유형"].eq("중개거래")].groupby(["시군구명", "월"])
                  .agg(중개_거래수=("gap", "size"), 중개_gap_중위=("gap", "median")))
            .reset_index())
gu_month["사용가능월"] = gu_month["월"] + 2
gu_month["월"] = gu_month["월"].dt.to_timestamp(); gu_month["사용가능월"] = gu_month["사용가능월"].dt.to_timestamp()
assert gu_month[["시군구명", "월"]].duplicated().sum() == 0 and gu_month["시군구명"].nunique() == 25
gu_month.to_parquet(MODEL_DIR / "oof_gap_gu_month.parquet", index=False)
print(gu_month.shape, gu_month["월"].min().date(), "~", gu_month["월"].max().date())
gu_month.pivot(index="월", columns="시군구명", values="중개_gap_중위").iloc[-6:, :8].round(3)

(1350, 8) 2022-01-01 ~ 2026-06-01


시군구명,강남구,강동구,강북구,강서구,관악구,광진구,구로구,금천구
월,,,,,,,,
2026-01-01,0.002,0.011,0.010,0.015,0.025,0.025,0.017,0.010
2026-02-01,0.018,0.036,0.027,0.042,0.041,0.033,0.021,0.011
2026-03-01,-0.015,0.027,0.029,0.038,0.028,0.002,0.025,0.001
2026-04-01,-0.037,0.006,0.039,0.015,0.014,0.009,0.013,0.004
2026-05-01,-0.018,0.015,0.025,0.020,0.014,0.022,0.009,0.004
2026-06-01,0.008,0.035,0.023,0.028,0.019,0.038,0.021,0.024


## 10. 최종 서비스 모델

2026-06까지 모든 거래로 학습한다 (2026-08 서비스 시점 기준 확정 거래). 설정 · 반복 수는 5-3장 매달 재학습 평가와 같다.
모델 파일은 gzip으로 저장한다 (GitHub 100MB 제한).

서비스용 오차 범위 표는 최근 3개 확정월(2026-04~06) OOF 잔차로 만든다 — 2026-08 예측에 쓰는 표다.

In [18]:
LGB_PARAMS = dict(final_params["lgb"], learning_rate=final_params["learning_rate"], n_estimators=final_params["n_estimators"],
                  bagging_freq=1, random_state=42, n_jobs=8, verbose=-1)
X = trade[FEATURES].copy()
for c in CATEGORICAL:
    X[c] = X[c].astype("category")
y = np.log(trade["거래금액"].to_numpy(float))
final = lgb.LGBMRegressor(**LGB_PARAMS).fit(X, y)
with gzip.open(MODEL_DIR / "lgbm_regression_v2.txt.gz", "wt", encoding="utf-8") as f:
    f.write(final.booster_.model_to_string())

# 저장한 파일로 다시 읽어 같은 예측이 나오는지 확인
with gzip.open(MODEL_DIR / "lgbm_regression_v2.txt.gz", "rt", encoding="utf-8") as f:
    reloaded = lgb.Booster(model_str=f.read())
sample = X.sample(2000, random_state=0)
assert np.allclose(reloaded.predict(sample), final.predict(sample))
print("✅ 저장 · 재로딩 예측 일치 |", round((MODEL_DIR / "lgbm_regression_v2.txt.gz").stat().st_size / 1e6, 1), "MB")

importance = pd.Series(final.booster_.feature_importance("gain"), index=FEATURES)
(importance / importance.sum() * 100).sort_values(ascending=False).head(20).round(2).to_frame("gain 비중 %")

✅ 저장 · 재로딩 예측 일치 | 5.4 MB


,gain 비중 %
직전거래가_동일면적,28.84
직전3건중위_동일면적,24.77
공시기반_추정가,21.83
직전거래가_시세보정,6.48
예상가격_단지만,5.24
전용면적,4.68
공시_m2당_단지,2.82
전용_중위,0.73
대형85초과_비중,0.37
공시_m2당_면적대,0.32


In [19]:
serve_month = pd.Period("2026-08", "M")
q, q_all = interval_table(serve_month, ALPHA, WINDOW)
serve_iv = q.reset_index()
serve_iv["적정가 하한 배수"] = np.exp(serve_iv["lo"]); serve_iv["적정가 상한 배수"] = np.exp(serve_iv["hi"])
serve_iv["등급 사용"] = serve_iv["단지"].eq("기존")
serve_iv.to_csv(MODEL_DIR / "interval_table.csv", index=False, encoding="utf-8-sig")

config = {
    "model": "LightGBM", "target": "log(거래금액)", "params": LGB_PARAMS,
    "features": FEATURES, "categorical": CATEGORICAL, "train_period": ["2021-07-01", "2026-06-30"],
    "service_input": {"거래유형": "중개거래", "예측 기준일": "서비스 월 1일", "확정 거래": "m-2월까지"},
    "interval": {"method": "OOF 잔차 분위수, 이동 창", "window_months": WINDOW, "alpha": ALPHA,
                 "segments": SEG, "price_band_억": [6, 9, 15], "fallback": "창 전체 분위수",
                 "table": "interval_table.csv", "table_month": str(serve_month)},
    "grade": {"rule": "구간 밖 초과분이 해당 방향 반폭 이하 → 다소, 초과 → 고/저평가", "new_complex": "등급 없음(참고용)"},
    "retrain": f"매달 (m-2월까지 확정 거래), 반복 {final_params['n_estimators']}회 고정",
}
json.dump(config, open(MODEL_DIR / "model_config.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)

metrics = {
    "rolling_monthly": {k: perf.loc[k, ["n", "MAPE", "편향"]].round(3).to_dict("index") for k in ["Valid", "Test"]},
    "rolling_overall": overall.round(3).to_dict("index"),
    "fixed_split_before_last_trade": {"Valid_MAPE": 6.160, "Test_MAPE": 9.637, "Test_MAPE_TrainValid_refit": 8.052},
    "vs_last_trade_baseline": compare.to_dict("records"),
    "naive_rule": {"Valid_MAPE": round(naive["Valid_MAPE"], 3), "Test_MAPE": round(naive["Test_MAPE"], 3)},
    "same_protocol_v1_vs_v2_test": {"V1": 13.26, "V2": 9.92},
    "interval_test": {"coverage_existing_pct": round(test_iv.loc[mk & test_iv["단지"].eq("기존"), "포함"].mean() * 100, 2),
                      "width_existing_pct": round(test_iv.loc[mk & test_iv["단지"].eq("기존"), "폭%"].mean(), 2),
                      "coverage_new_pct": round(test_iv.loc[mk & test_iv["단지"].eq("신규"), "포함"].mean() * 100, 2)},
}
json.dump(metrics, open(MODEL_DIR / "metrics.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=float)
test_iv.merge(trade[["transaction_id", "단지일련번호", "시군구명", "거래금액"]], on="transaction_id").assign(
    예측가_만원=lambda d: np.exp(d["p"]).round(0)).drop(columns=["y"]).to_parquet(MODEL_DIR / "test_predictions.parquet", index=False)
display(serve_iv.round(3))
print(sorted(p.name for p in MODEL_DIR.iterdir()))

,단지,가격대,lo,hi,적정가 하한 배수,적정가 상한 배수,등급 사용
0,기존,15억 이상,-0.090,0.078,0.914,1.081,True
1,기존,6~9억,-0.056,0.105,0.945,1.111,True
2,기존,6억 미만,-0.082,0.103,0.921,1.108,True
3,기존,9~15억,-0.057,0.108,0.945,1.114,True
4,신규,15억 이상,-0.187,0.485,0.829,1.624,False
5,신규,6~9억,-0.245,0.212,0.783,1.236,False
6,신규,6억 미만,-0.106,0.225,0.899,1.253,False
7,신규,9~15억,-0.391,0.356,0.677,1.428,False


['interval_table.csv', 'lgbm_regression_v2.txt.gz', 'metrics.json', 'model_config.json', 'oof_gap_gu_month.parquet', 'oof_predictions.parquet', 'test_predictions.parquet']


## 11. 정리

| 항목 | 결정 |
|---|---|
| 모델 | LightGBM 단일 (Optuna, 시간 fold 2개, 잎 72 · 반복 2,080), 정답 `log(거래금액)`, 모델 파일 5.4MB (gzip) |
| Feature | 레지스트리 기본 201개. 예상가격은 단지 자체 이력일 때만 (`예상가격_단지만`), 같은 단지 · 면적 직전 거래 5개 + 구 주간 지수 시세 보정 2개 (`06_last_trade`) |
| 보완 2 후보 | 상권 · 배정 초등학교 · K-apt 단지 속성은 본 모델 설정 재비교에서 불채택 → 화면 표시값 |
| 앙상블 · 정답 변환 · 최근 오차 보정 | 채택 기준 미달 → 미사용 |
| 평가 | 매달 재학습 OOF: Valid 5.25% · Test 5.65% (중개 4.86 · 5.35%, 5-2 모델 4.95 · 5.53%). 직전 거래 있는 Test 중개: 모델 5.19% vs 직전 거래가 × 시세 보정 6.23% vs 직전 거래가 6.67%, 단순 규칙 14.94% |
| 오차 범위 | 최근 3개 확정월 OOF 잔차(창 3개월 고정), 단지 × 가격대별 8~92% 분위수 → 기존 단지 Test 포함률 81.8%, 평균 폭 16.4% |
| 등급 | 범위 기준 5단계 규칙, 신규 단지는 등급 없음 |
| 31 입력 | `oof_gap_gu_month.parquet` (사용가능월 = 거래월 + 2) |

### 남은 한계와 처리 계획

| 한계 | 영향 | 처리 |
|---|---|---|
| 상승기 과소추정 (Test 편향 -1.5%) | 동작 · 동대문 · 강북 등 상승 지역 적정가가 2~3% 낮게 나옴 | 확정 거래 2개월 지연은 구조적 한계. 웹에 "최근 2개월 거래 미반영" 안내, 31 지역 흐름(4주 흐름 순위)과 함께 표시 |
| 신규 단지 구간 | 80% 미달 | 등급 없이 참고용 범위 (설계서 6-1 갱신) |
| 직거래 | 오차 2배, 저평가 판정 많음 | 서비스 입력은 중개거래 기준, 거래 목록에서 직거래 구분 표시 |
| 급락장 · 거래 절벽 | 2022 하반기 OOF MAPE 12.5% (시세 보정 전 12.9%) — 구 지수는 구 평균이라 단지별 낙폭을 못 따라감 | 직전 거래 경과일을 화면에 표시, 31 하락 신호와 함께 해석 |
| 2022 OOF 품질 | 초기 학습 데이터 짧음 (중개 MAPE 8.7%) | 31에서 Gap Feature Ablation |
| CatBoost 재현 불안정 | 후보 제외 | V3에서 필요 시 재검토 |
| 응급 · 유치원 거리 (현재 목록) | ablation에서 효과 없음 | 기본 Feature 제외 유지 |